## Legal Chat Bot

### PART A: Divide our documents into chunks

In [ ]:
import os
import glob
import tiktoken
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# price is a factor for testing, so we're going to use a low cost model
load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
db_name = "chat_vector_db"
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# How many characters in all the documents?
knowledge_base_path = "knowledge-base/*"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
# How many tokens in all the documents?
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders
documents = []
loader = DirectoryLoader("knowledge-base", glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
folder_docs = loader.load()
for doc in folder_docs:
    documents.append(doc)    

print(f"Loaded {len(documents)} documents")

In [ ]:
documents[1]

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,          # Larger chunks preserve more section context
    chunk_overlap=300,         # More overlap to avoid splitting mid-provision
    separators=[
        "\n## ",              # Major headings (Parts/Chapters)
        "\n### ",             # Sub-headings (Sections)
        "\n\n",               # Paragraph breaks
        "\n",                 # Line breaks
        ". ",                 # Sentences
        " ",                  # Words
    ]
)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

## Add Metadata to Chunks
After splitting, enrich each chunk with source metadata:

In [ ]:
for chunk in chunks:
    # Extract Act name and section from content or filename
    source_file = chunk.metadata.get("source", "")
    if "consumer" in source_file.lower():
        chunk.metadata["act"] = "FCCPA 2018"
    elif "1999" in source_file:
        chunk.metadata["act"] = "Constitution 1999"

### PART B: Make vectors and store in Chroma

Remember to set up a Hugging Face account and get your HF_TOKEN

Add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [ ]:
# Pick an embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 8,                        # Retrieve more candidates
        "score_threshold": 0.3,         # Filter out low-relevance chunks
    }
)

llm = ChatOpenAI(temperature=0, model_name=MODEL, base_url="https://openrouter.ai/api/v1")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a Nigerian legal information assistant. You help users understand their legal rights 
under Nigerian law by citing specific statutes, sections, and case precedents.

RULES:
1. ONLY cite laws, sections, and cases that appear in the provided context below.
2. Structure your response as: (a) applicable law, (b) relevant sections with quotes, 
   (c) practical next steps the user can take. (d) Expected outcome (e) Timeline
3. If the context does not contain relevant law for the question, say: 
   "The documents I have access to do not cover this area of law."
4. Always end with: "Disclaimer: This is legal information, not legal advice. 
   Please consult a qualified Nigerian lawyer for advice specific to your situation."
Keep it short, concise, and straight to the point.

Context:
{context}
"""

In [ ]:
def expand_query(question: str) -> str:
    """Use LLM to add legal terms to the query for better retrieval."""
    expansion_prompt = f"""Given this user question about Nigerian law, 
    rewrite it to include relevant legal terminology that would help 
    find matching statutes. Keep it concise.
    
    Question: {question}
    Rewritten query:"""
    response = llm.invoke([HumanMessage(content=expansion_prompt)])
    return response.content

In [ ]:
llm_question = expand_query("I bought a phone in Nigeria that broke after 1 week. Shop won't refund. What are my rights?")
llm_question

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer = answer_question(llm_question, [])
print(answer)